# 🎬 Scalable Video Recommendation System
## Step 1: Enterprise-Grade Data Pipeline
### 数据集: MovieLens-1M（流程验证版）

---
**架构对应关系**
```
MovieLens ratings.dat   →  对应 KuaiRec big_matrix.csv      （用户-视频交互）
MovieLens users.dat     →  对应 KuaiRec user_features.csv   （用户画像）
MovieLens movies.dat    →  对应 KuaiRec item_categories.csv （视频元数据）
```

**本 Notebook 完成**:
```
Cell 1  → 安装依赖
Cell 2  → 全局配置 & 目录结构（模拟 OSS Bucket）
Cell 3  → 企业级 Logger & Timer
Cell 4  → 数据下载（自动，无需手动操作）
Cell 5  → 数据加载 & 格式解析
Cell 6  → 数据质量检验 DQC
Cell 7  → EDA 可视化报告
Cell 8  → 数据清洗 Pipeline
Cell 9  → User 侧特征工程
Cell 10 → Item 侧特征工程
Cell 11 → 用户行为序列特征
Cell 12 → 样本工厂（正负采样 + 时序划分 + 穿越防护）
Cell 13 → Feature Store 写入 & 验证
Cell 14 → Pipeline 健康度报告（PSI 监控）
```

> ✅ **一键运行**: 点击 `运行时` → `全部运行`，无需任何手动操作

---
## Cell 1 — 安装依赖

In [ ]:
# ============================================================
# 安装企业级推荐系统工具栈
# ============================================================
!pip install -q pandas numpy scikit-learn pyarrow fastparquet
!pip install -q scipy tqdm tabulate matplotlib seaborn

print('✅ 所有依赖安装完成')

---
## Cell 2 — 全局配置 & 目录结构

In [ ]:
# ============================================================
# 全局配置
# 企业对应: 配置中心（Apollo / Nacos）
# ============================================================
import os
import warnings
warnings.filterwarnings('ignore')

# ---------- 选择存储位置 ----------
# USE_GDRIVE = True  → 数据存到 Google Drive（永久保存，推荐）
# USE_GDRIVE = False → 数据存到 Colab 本地（Session 结束后消失）
USE_GDRIVE = True  # 第一次运行先设 False，跑通后改 True

if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/VideoRecSys'
else:
    BASE_DIR = '/content/VideoRecSys'

# ---------- 目录结构（模拟 OSS 分层）----------
PATHS = {
    'raw'          : f'{BASE_DIR}/data/raw',
    'cleaned'      : f'{BASE_DIR}/data/cleaned',
    'features'     : f'{BASE_DIR}/data/features',
    'samples'      : f'{BASE_DIR}/data/samples',
    'feature_store': f'{BASE_DIR}/feature_store',
    'logs'         : f'{BASE_DIR}/logs',
    'reports'      : f'{BASE_DIR}/reports',
}

for path in PATHS.values():
    os.makedirs(path, exist_ok=True)

print('✅ 目录结构创建完成')
print(f'   存储根目录: {BASE_DIR}')
for k, v in PATHS.items():
    print(f'   {k:15s} → {v}')

---
## Cell 3 — 企业级 Logger & Timer

In [ ]:
# ============================================================
# 企业级日志 & 计时工具
# 对应: 阿里云 SLS 日志服务 / ELK
# ============================================================
import logging
import time
from datetime import datetime

def get_logger(name='RecSysPipeline'):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
    # Console
    ch = logging.StreamHandler()
    fmt = logging.Formatter('[%(asctime)s][%(levelname)s] %(message)s', '%H:%M:%S')
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    # File
    log_file = f"{PATHS['logs']}/pipeline_{datetime.now().strftime('%Y%m%d')}.log"
    fh = logging.FileHandler(log_file)
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

logger = get_logger()

class Timer:
    """任务计时器 → 对应企业监控的 Task Duration 指标"""
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.start = time.time()
        logger.info(f'▶  开始: {self.name}')
        return self
    def __exit__(self, *args):
        elapsed = time.time() - self.start
        logger.info(f'✅ 完成: {self.name}  耗时 {elapsed:.2f}s')

logger.info('Logger 初始化完成 ✓')

---
## Cell 4 — 数据下载（自动）

In [ ]:
# ============================================================
# MovieLens-1M 数据集下载
# 官方地址: https://grouplens.org/datasets/movielens/1m/
# 大小: ~6MB（压缩）
# 内容: 100万条评分 + 6040用户 + 3952部电影
#
# 企业对应: 从上游业务方 OSS/S3 拉取原始日志
# ============================================================
import urllib.request
import zipfile

ML_URL  = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
ZIP_PATH = f"{PATHS['raw']}/ml-1m.zip"
ML_DIR   = f"{PATHS['raw']}/ml-1m"

with Timer('MovieLens-1M 数据下载'):
    if not os.path.exists(ML_DIR):
        logger.info('  正在下载 ml-1m.zip (~6MB)...')
        urllib.request.urlretrieve(ML_URL, ZIP_PATH)
        logger.info('  解压中...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PATHS['raw'])
        os.remove(ZIP_PATH)
        logger.info(f'  解压完成 → {ML_DIR}')
    else:
        logger.info('  ⏭  已存在，跳过下载')

# 列出文件
print('\n【原始文件列表】')
for f in os.listdir(ML_DIR):
    size = os.path.getsize(f'{ML_DIR}/{f}') / 1024
    print(f'  {f:25s}  {size:.1f} KB')

---
## Cell 5 — 数据加载 & 格式解析

In [ ]:
# ============================================================
# 加载 MovieLens-1M 原始数据
# ML-1M 使用 :: 分隔符，需要手动指定列名
# ============================================================
import pandas as pd
import numpy as np

with Timer('数据加载'):

    # --- 1. 评分数据（核心交互表）---
    # 对应 KuaiRec: big_matrix.csv
    # UserID :: MovieID :: Rating :: Timestamp
    df_ratings = pd.read_csv(
        f'{ML_DIR}/ratings.dat',
        sep='::',
        header=None,
        names=['user_id', 'item_id', 'rating', 'timestamp'],
        engine='python',
        encoding='latin-1'
    )

    # --- 2. 用户特征表 ---
    # 对应 KuaiRec: user_features.csv
    # UserID :: Gender :: Age :: Occupation :: Zip-code
    df_users = pd.read_csv(
        f'{ML_DIR}/users.dat',
        sep='::',
        header=None,
        names=['user_id', 'gender', 'age', 'occupation', 'zip_code'],
        engine='python',
        encoding='latin-1'
    )

    # --- 3. 电影元数据表 ---
    # 对应 KuaiRec: item_categories.csv
    # MovieID :: Title :: Genres
    df_movies = pd.read_csv(
        f'{ML_DIR}/movies.dat',
        sep='::',
        header=None,
        names=['item_id', 'title', 'genres'],
        engine='python',
        encoding='latin-1'
    )

    # timestamp 转为可读时间
    df_ratings['datetime'] = pd.to_datetime(df_ratings['timestamp'], unit='s')

logger.info(f'评分数据: {len(df_ratings):,} 行')
logger.info(f'用户数量: {df_ratings["user_id"].nunique():,}')
logger.info(f'电影数量: {df_ratings["item_id"].nunique():,}')
logger.info(f'时间范围: {df_ratings["datetime"].min()} ~ {df_ratings["datetime"].max()}')

print('\n【评分数据样例】')
print(df_ratings.head())
print('\n【用户数据样例】')
print(df_users.head())
print('\n【电影数据样例】')
print(df_movies.head())

---
## Cell 6 — 数据质量检验 DQC

In [ ]:
# ============================================================
# 企业级数据质量检验（DQC）
# 对应: 阿里 DataWorks 数据质量平台
# 规则: 缺失值 / 重复值 / 异常值 / 分布检验
# ============================================================
from tabulate import tabulate

def dqc_report(df, name, key_cols=None):
    print(f'\n{"="*60}')
    print(f'  📊 DQC 报告: {name}')
    print(f'{"="*60}')

    # 基础信息
    print(f'  行数: {len(df):,}  列数: {df.shape[1]}  '
          f'内存: {df.memory_usage(deep=True).sum()/1024**2:.2f} MB')

    # 缺失值
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    rows = []
    for col in df.columns:
        pct = missing_pct[col]
        status = '❌ 告警' if pct > 5 else ('⚠️  注意' if pct > 0 else '✅ 正常')
        rows.append([col, missing[col], f'{pct}%', status])
    print('\n  【缺失值检验】')
    print(tabulate(rows, headers=['列名','缺失数','缺失率','状态'],
                   tablefmt='simple', showindex=False))

    # 重复值
    dup = df.duplicated(subset=key_cols).sum()
    print(f'\n  【重复行检验】 重复数: {dup:,}  '
          + ('❌ 告警' if dup > 0 else '✅ 正常'))

    # 数值列范围
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        print('\n  【数值列统计】')
        print(df[num_cols].describe().round(2).to_string())

dqc_report(df_ratings, '评分数据 ratings', key_cols=['user_id','item_id'])
dqc_report(df_users,   '用户数据 users',   key_cols=['user_id'])
dqc_report(df_movies,  '电影数据 movies',  key_cols=['item_id'])

---
## Cell 7 — EDA 可视化报告

In [ ]:
# ============================================================
# 探索性数据分析（EDA）
# 企业对应: 每日/每周数据分析报告
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.style.use('dark_background')
fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('#0d1117')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
ACCENT = '#00d4ff'

def styled_ax(ax, title):
    ax.set_facecolor('#161b22')
    ax.set_title(title, color='#c9d1d9', fontsize=11, pad=10)
    ax.tick_params(colors='#6e7681')
    for spine in ax.spines.values(): spine.set_color('#30363d')
    return ax

# 1. 评分分布
ax1 = styled_ax(fig.add_subplot(gs[0,0]), 'Rating Distribution（评分分布）')
counts = df_ratings['rating'].value_counts().sort_index()
bars = ax1.bar(counts.index, counts.values, color=ACCENT, alpha=0.85, width=0.6)
ax1.set_xlabel('Rating', color='#6e7681')
ax1.set_ylabel('Count', color='#6e7681')
for bar in bars:
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5000,
             f'{int(bar.get_height()/1000)}k', ha='center', color='#c9d1d9', fontsize=8)

# 2. 用户交互次数（长尾分布）
ax2 = styled_ax(fig.add_subplot(gs[0,1]), 'User Interactions（长尾分布）')
user_counts = df_ratings['user_id'].value_counts()
ax2.hist(user_counts, bins=50, color='#39d353', alpha=0.8, edgecolor='#0d1117')
ax2.set_xlabel('# Interactions per User', color='#6e7681')
ax2.set_yscale('log')
ax2.axvline(user_counts.median(), color='#f0c040', lw=1.5, linestyle='--',
            label=f'Median={user_counts.median():.0f}')
ax2.legend(fontsize=8)

# 3. 电影热度分布
ax3 = styled_ax(fig.add_subplot(gs[0,2]), 'Item Popularity（热度分布）')
item_counts = df_ratings['item_id'].value_counts()
ax3.hist(item_counts, bins=50, color='#ff8c42', alpha=0.8, edgecolor='#0d1117')
ax3.set_xlabel('# Interactions per Item', color='#6e7681')
ax3.set_yscale('log')

# 4. 用户年龄分布
ax4 = styled_ax(fig.add_subplot(gs[1,0]), 'User Age Distribution（年龄分布）')
age_map = {1:'<18',18:'18-24',25:'25-34',35:'35-44',45:'45-49',50:'50-55',56:'56+'}
df_users['age_label'] = df_users['age'].map(age_map)
age_order = ['<18','18-24','25-34','35-44','45-49','50-55','56+']
age_counts = df_users['age_label'].value_counts().reindex(age_order)
ax4.bar(range(len(age_counts)), age_counts.values, color='#b48eff', alpha=0.85)
ax4.set_xticks(range(len(age_counts)))
ax4.set_xticklabels(age_order, rotation=30, fontsize=8, color='#6e7681')

# 5. 用户活跃度分层（冷/热用户）
ax5 = styled_ax(fig.add_subplot(gs[1,1]), 'User Activity Tier（活跃度分层）')
bins = [0, 20, 50, 100, 200, 9999]
labels = ['极冷\n<20', '冷\n20-50', '中\n50-100', '活跃\n100-200', '高活跃\n>200']
tier = pd.cut(user_counts, bins=bins, labels=labels)
tier_counts = tier.value_counts().reindex(labels)
colors = ['#ff5757','#ff8c42','#f0c040','#39d353','#00d4ff']
ax5.bar(range(len(tier_counts)), tier_counts.values, color=colors, alpha=0.85)
ax5.set_xticks(range(len(tier_counts)))
ax5.set_xticklabels(labels, fontsize=8, color='#6e7681')

# 6. 评分时间趋势
ax6 = styled_ax(fig.add_subplot(gs[1,2]), 'Rating Volume by Month（月度趋势）')
df_ratings['month'] = df_ratings['datetime'].dt.to_period('M')
monthly = df_ratings.groupby('month').size()
ax6.plot(range(len(monthly)), monthly.values, color=ACCENT, lw=2)
ax6.fill_between(range(len(monthly)), monthly.values, alpha=0.15, color=ACCENT)
ax6.set_xlabel('Month', color='#6e7681')
ax6.set_xticks([])

fig.suptitle('MovieLens-1M EDA Report', color='#c9d1d9', fontsize=15, fontweight='bold', y=0.98)
plt.savefig(f"{PATHS['reports']}/eda_report.png", dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
logger.info('EDA 报告已保存')

---
## Cell 8 — 数据清洗 Pipeline

In [ ]:
# ============================================================
# 企业级数据清洗 Pipeline
# 对应: 阿里 DataWorks 清洗任务 / Flink 实时清洗
# 每步记录过滤量 → 对应数据血缘追踪
# ============================================================

class DataCleaningPipeline:
    def __init__(self, df, name):
        self.df = df.copy()
        self.name = name
        self.initial_count = len(df)
        self.log = []
        logger.info(f'[{name}] 初始数据量: {self.initial_count:,}')

    def _record(self, step, before, after):
        n = before - after
        pct = n / before * 100 if before > 0 else 0
        self.log.append({'步骤': step, '过滤前': before,
                         '过滤后': after, '过滤量': n, '过滤率': f'{pct:.2f}%'})
        logger.info(f'  [{step}] {before:,} → {after:,}  过滤 {n:,} ({pct:.2f}%)')
        return self

    def remove_duplicates(self, subset):
        b = len(self.df)
        self.df = self.df.drop_duplicates(subset=subset)
        return self._record('去重', b, len(self.df))

    def filter_missing(self, cols):
        b = len(self.df)
        self.df = self.df.dropna(subset=cols)
        return self._record('过滤缺失值', b, len(self.df))

    def filter_rating_range(self, min_r=1, max_r=5):
        b = len(self.df)
        self.df = self.df[(self.df['rating'] >= min_r) & (self.df['rating'] <= max_r)]
        return self._record(f'过滤异常评分[{min_r},{max_r}]', b, len(self.df))

    def filter_cold_users(self, min_count=20):
        b = len(self.df)
        counts = self.df['user_id'].value_counts()
        valid = counts[counts >= min_count].index
        self.df = self.df[self.df['user_id'].isin(valid)]
        return self._record(f'过滤冷用户(<{min_count}次)', b, len(self.df))

    def filter_cold_items(self, min_count=10):
        b = len(self.df)
        counts = self.df['item_id'].value_counts()
        valid = counts[counts >= min_count].index
        self.df = self.df[self.df['item_id'].isin(valid)]
        return self._record(f'过滤冷视频(<{min_count}次)', b, len(self.df))

    def add_label(self, threshold=4):
        """rating >= threshold 为正样本（对应 KuaiRec 的 watch_ratio >= 0.7）"""
        self.df['label'] = (self.df['rating'] >= threshold).astype(int)
        pos_rate = self.df['label'].mean()
        logger.info(f'  [生成标签] 正样本率: {pos_rate:.2%}  (rating >= {threshold})')
        return self

    def sort_by_time(self):
        """按时间排序 → 为后续时序划分做准备（重要！）"""
        self.df = self.df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
        logger.info('  [时序排序] 按 user_id + timestamp 排序完成')
        return self

    def report(self):
        print(f'\n{"="*58}')
        print(f'  🧹 清洗报告: {self.name}')
        print(f'{"="*58}')
        print(tabulate(self.log, headers='keys', tablefmt='simple'))
        total = self.initial_count - len(self.df)
        print(f'\n  总计过滤: {total:,} 条 ({total/self.initial_count:.2%})')
        print(f'  最终数据: {len(self.df):,} 条')
        return self

    def save(self, path):
        self.df.to_parquet(path, index=False, compression='snappy')
        mb = os.path.getsize(path) / 1024**2
        logger.info(f'  保存 → {path} ({mb:.2f} MB, Parquet/Snappy)')
        return self


with Timer('数据清洗 Pipeline'):
    pipe = DataCleaningPipeline(df_ratings, '评分数据')
    cleaned_df = (
        pipe
        .remove_duplicates(subset=['user_id', 'item_id'])
        .filter_missing(cols=['user_id', 'item_id', 'rating'])
        .filter_rating_range(min_r=1, max_r=5)
        .filter_cold_users(min_count=20)
        .filter_cold_items(min_count=10)
        .add_label(threshold=4)
        .sort_by_time()
        .report()
        .save(f"{PATHS['cleaned']}/interactions_cleaned.parquet")
        .df
    )

---
## Cell 9 — User 侧特征工程

In [ ]:
# ============================================================
# 用户侧特征工程
# 对应: 用户画像平台（User Profile Service）
# 特征分类:
#   - 统计特征（行为历史计算）
#   - 人口统计特征（性别、年龄、职业）
#   - 活跃度特征（分层标签）
# ============================================================

with Timer('User 侧特征工程'):

    # ---- 1. 行为统计特征 ----
    user_stats = cleaned_df.groupby('user_id').agg(
        interaction_count = ('item_id',   'count'),
        avg_rating        = ('rating',    'mean'),
        std_rating        = ('rating',    'std'),
        min_rating        = ('rating',    'min'),
        max_rating        = ('rating',    'max'),
        positive_rate     = ('label',     'mean'),   # 用户正反馈率
        unique_items      = ('item_id',   'nunique'),
        first_active      = ('timestamp', 'min'),
        last_active       = ('timestamp', 'max'),
    ).reset_index()

    user_stats['std_rating'] = user_stats['std_rating'].fillna(0)

    # ---- 2. 活跃周期（天数）----
    user_stats['active_days'] = (
        (user_stats['last_active'] - user_stats['first_active']) / 86400
    ).round(1)

    # ---- 3. 用户活跃度分层 ----
    def activity_tier(n):
        if n < 20:   return 0   # 极冷
        elif n < 50: return 1   # 冷
        elif n < 100:return 2   # 普通
        elif n < 200:return 3   # 活跃
        else:        return 4   # 高活跃
    user_stats['activity_tier'] = user_stats['interaction_count'].apply(activity_tier)

    # ---- 4. 合并人口统计特征 ----
    # 性别编码: M→1, F→0
    df_users['gender_enc'] = (df_users['gender'] == 'M').astype(int)
    user_feat_cols = ['user_id', 'gender_enc', 'age', 'occupation']
    user_features = pd.merge(user_stats, df_users[user_feat_cols], on='user_id', how='left')

    # 保存
    user_features.to_parquet(
        f"{PATHS['features']}/user_features.parquet",
        index=False, compression='snappy'
    )

logger.info(f'用户特征: {user_features.shape[0]:,} 用户  {user_features.shape[1]} 个特征')
print('\n【特征列表】')
for col in user_features.columns:
    print(f'  {col:25s}  {user_features[col].dtype}')
print('\n【样例】')
print(user_features.head(3).to_string())

---
## Cell 10 — Item 侧特征工程

In [ ]:
# ============================================================
# Item 侧特征工程
# 对应: 视频内容理解平台（Content Understanding）
# 特征分类:
#   - 统计特征（播放量、点击率）
#   - 内容特征（类型、年份）
#   - 热度特征（分层标签）
# ============================================================

with Timer('Item 侧特征工程'):

    # ---- 1. 视频统计特征 ----
    item_stats = cleaned_df.groupby('item_id').agg(
        play_count    = ('user_id', 'count'),
        avg_rating    = ('rating',  'mean'),
        std_rating    = ('rating',  'std'),
        ctr_proxy     = ('label',   'mean'),   # 正反馈率 → 对应 CTR
        unique_users  = ('user_id', 'nunique'),
    ).reset_index()
    item_stats['std_rating'] = item_stats['std_rating'].fillna(0)

    # ---- 2. 热度分层（5层分位数）----
    item_stats['popularity_tier'] = pd.qcut(
        item_stats['play_count'], q=5, labels=[0,1,2,3,4], duplicates='drop'
    ).astype(int)

    # ---- 3. 解析电影元数据 ----
    # 提取上映年份
    df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)').astype(float)

    # 多热编码 genres（Action|Comedy|Drama → 多列 0/1）
    all_genres = set()
    df_movies['genres'].str.split('|').apply(all_genres.update)
    all_genres.discard('(no genres listed)')
    for g in sorted(all_genres):
        df_movies[f'genre_{g}'] = df_movies['genres'].str.contains(g).astype(int)
    genre_cols = [f'genre_{g}' for g in sorted(all_genres)]

    # 合并
    meta_cols = ['item_id', 'year'] + genre_cols
    item_features = pd.merge(item_stats, df_movies[meta_cols], on='item_id', how='left')
    item_features['year'] = item_features['year'].fillna(item_features['year'].median())

    # 保存
    item_features.to_parquet(
        f"{PATHS['features']}/item_features.parquet",
        index=False, compression='snappy'
    )

logger.info(f'视频特征: {item_features.shape[0]:,} 个视频  {item_features.shape[1]} 个特征')
logger.info(f'Genre 特征数: {len(genre_cols)}')
print('\n【样例】')
print(item_features.head(3).to_string())

---
## Cell 11 — 用户行为序列特征（DIN 模型核心输入）

In [ ]:
# ============================================================
# 用户行为序列特征
# 这是 DIN / DIEN / BST 等序列模型的核心输入
# 对应: 实时序列服务（Sequence Feature Service）
#
# 关键细节:
#   - 序列必须按时间排序（已在 Cell 8 完成）
#   - 保留最近 SEQ_LEN 个行为
#   - 正序列（喜欢的）和全序列分开存
# ============================================================

SEQ_LEN = 50  # 企业标准：保留最近 50 个行为

with Timer('用户行为序列特征'):
    seq_records = []

    for user_id, group in cleaned_df.groupby('user_id'):
        # 已按时间排序（Cell 8 保证）
        all_items = group['item_id'].tolist()
        pos_items = group[group['label'] == 1]['item_id'].tolist()

        seq_records.append({
            'user_id'    : user_id,
            'all_seq'    : ','.join(map(str, all_items[-SEQ_LEN:])),
            'pos_seq'    : ','.join(map(str, pos_items[-SEQ_LEN:])),
            'seq_len'    : min(len(all_items), SEQ_LEN),
            'pos_seq_len': min(len(pos_items), SEQ_LEN),
        })

    seq_df = pd.DataFrame(seq_records)
    seq_df.to_parquet(
        f"{PATHS['features']}/user_sequences.parquet",
        index=False, compression='snappy'
    )

logger.info(f'序列特征: {len(seq_df):,} 用户  序列上限: {SEQ_LEN}')
logger.info(f'平均序列长度: {seq_df["seq_len"].mean():.1f}')
print('\n【样例】')
print(seq_df.head(3).to_string())

---
## Cell 12 — 样本工厂（正负采样 + 时序划分 + 穿越防护）

In [ ]:
# ============================================================
# 企业级样本工厂
# 对应: 阿里 Sample Factory
#
# 三大核心原则:
# 1. 时序划分（禁止随机 split，否则造成数据穿越）
# 2. 负样本采样（随机 + 热门混合，比例 1:4）
# 3. 特征拼接（在划分后的集合上分别 join）
# ============================================================

class SampleFactory:
    def __init__(self, interactions, user_feat, item_feat, user_seq):
        self.df       = interactions
        self.u_feat   = user_feat.set_index('user_id')
        self.i_feat   = item_feat.set_index('item_id')
        self.seq      = user_seq.set_index('user_id')
        self.all_items = interactions['item_id'].unique()
        # 预计算热门权重（用于热门负采样）
        pop = interactions['item_id'].value_counts(normalize=True)
        self.hot_items   = pop.index.tolist()[:2000]
        self.hot_weights = pop.values[:2000]
        self.hot_weights = self.hot_weights / self.hot_weights.sum()

    def temporal_split(self, test_ratio=0.2):
        """
        时序划分 — 每个用户前 80% 训练，后 20% 测试
        ⚠️  企业铁律：绝不随机划分，防止未来信息泄露
        """
        train_parts, test_parts = [], []
        for uid, grp in self.df.groupby('user_id'):
            n = len(grp)
            cut = max(1, int(n * (1 - test_ratio)))
            train_parts.append(grp.iloc[:cut])
            test_parts.append(grp.iloc[cut:])
        self.train = pd.concat(train_parts, ignore_index=True)
        self.test  = pd.concat(test_parts,  ignore_index=True)
        logger.info(f'时序划分 → 训练: {len(self.train):,}  测试: {len(self.test):,}')
        return self

    def negative_sample(self, df, neg_ratio=4):
        """
        负样本采样: 50% 随机 + 50% 热门
        neg_ratio: 每个正样本配 N 个负样本（企业标准 1:4）
        """
        pos_df  = df[df['label'] == 1].copy()
        neg_rows = []

        # 每个用户已有交互的 item 集合（用于排除）
        user_seen = df.groupby('user_id')['item_id'].apply(set).to_dict()

        for _, row in pos_df.iterrows():
            uid  = row['user_id']
            seen = user_seen.get(uid, set())
            count, tries = 0, 0

            while count < neg_ratio and tries < neg_ratio * 15:
                # 交替随机采样 / 热门采样
                if tries % 2 == 0:
                    neg = int(np.random.choice(self.all_items))
                else:
                    neg = int(np.random.choice(self.hot_items, p=self.hot_weights))
                if neg not in seen:
                    neg_rows.append({
                        'user_id': uid, 'item_id': neg,
                        'rating': 0.0, 'timestamp': row['timestamp'],
                        'label': 0
                    })
                    count += 1
                tries += 1

        neg_df = pd.DataFrame(neg_rows)
        result = pd.concat([pos_df, neg_df], ignore_index=True)
        result = result.sample(frac=1, random_state=42).reset_index(drop=True)
        logger.info(f'  负采样: {len(pos_df):,} 正样本 + {len(neg_df):,} 负样本  '
                    f'正样本率: {result["label"].mean():.2%}')
        return result

    def join_features(self, df):
        """拼接用户 / 视频 / 序列特征"""
        df = df.join(self.u_feat, on='user_id', rsuffix='_u')
        df = df.join(self.i_feat, on='item_id', rsuffix='_i')
        seq_cols = [c for c in ['all_seq','pos_seq','seq_len','pos_seq_len']
                    if c in self.seq.columns]
        if seq_cols:
            df = df.join(self.seq[seq_cols], on='user_id')
        return df


with Timer('样本工厂'):
    factory = SampleFactory(cleaned_df, user_features, item_features, seq_df)
    factory.temporal_split(test_ratio=0.2)

    logger.info('构建训练集...')
    train_samples = factory.negative_sample(factory.train, neg_ratio=4)
    train_samples = factory.join_features(train_samples)

    logger.info('构建测试集...')
    test_samples = factory.join_features(factory.test)

    train_samples.to_parquet(f"{PATHS['samples']}/train.parquet", index=False, compression='snappy')
    test_samples.to_parquet( f"{PATHS['samples']}/test.parquet",  index=False, compression='snappy')

logger.info(f'训练集: {len(train_samples):,} 行  特征: {train_samples.shape[1]} 列')
logger.info(f'测试集: {len(test_samples):,} 行')
print('\n【训练集列名】')
print(list(train_samples.columns))

---
## Cell 13 — Feature Store 写入 & 在线读取验证

In [ ]:
# ============================================================
# 模拟企业级 Feature Store
# 生产环境: Redis（在线层）+ Hive/OSS（离线层）
# 本地模拟: Python dict（模拟 Redis Hash）
#
# 生产替换方式:
#   online_store[key] = val  →  redis.hset(key, mapping=val)
#   online_store[key]        →  redis.hgetall(key)
# ============================================================
import pickle

class FeatureStore:
    def __init__(self, path):
        self.path  = path
        self.store = {}   # 模拟 Redis
        os.makedirs(path, exist_ok=True)

    def _numeric_dict(self, row, exclude):
        return {
            k: round(float(v), 6)
            for k, v in row.items()
            if k not in exclude and isinstance(v, (int, float, np.integer, np.floating))
            and not np.isnan(float(v))
        }

    def write_users(self, df):
        for _, row in df.iterrows():
            self.store[f'user:{row["user_id"]}'] = self._numeric_dict(row, {'user_id'})
        df.to_parquet(f'{self.path}/user_features.parquet', index=False, compression='snappy')
        logger.info(f'  写入用户特征: {len(df):,} 条')

    def write_items(self, df):
        for _, row in df.iterrows():
            self.store[f'item:{row["item_id"]}'] = self._numeric_dict(row, {'item_id'})
        df.to_parquet(f'{self.path}/item_features.parquet', index=False, compression='snappy')
        logger.info(f'  写入视频特征: {len(df):,} 条')

    def write_sequences(self, df):
        for _, row in df.iterrows():
            self.store[f'seq:{row["user_id"]}'] = {
                'all_seq' : row.get('all_seq', ''),
                'pos_seq' : row.get('pos_seq', ''),
                'seq_len' : int(row.get('seq_len', 0)),
            }
        df.to_parquet(f'{self.path}/user_sequences.parquet', index=False, compression='snappy')
        logger.info(f'  写入序列特征: {len(df):,} 条')

    def get_user(self, uid):   return self.store.get(f'user:{uid}', {})
    def get_item(self, iid):   return self.store.get(f'item:{iid}', {})
    def get_seq(self, uid):
        s = self.store.get(f'seq:{uid}', {})
        if s.get('all_seq'): s['all_seq'] = list(map(int, s['all_seq'].split(',')))
        return s

    def snapshot(self):
        p = f'{self.path}/snapshot.pkl'
        with open(p, 'wb') as f: pickle.dump(self.store, f)
        mb = os.path.getsize(p) / 1024**2
        logger.info(f'  快照保存 → {p} ({mb:.2f} MB)  总 key: {len(self.store):,}')


with Timer('Feature Store 写入'):
    fs = FeatureStore(PATHS['feature_store'])
    fs.write_users(user_features)
    fs.write_items(item_features)
    fs.write_sequences(seq_df)
    fs.snapshot()

# ---- 在线读取验证 ----
sample_uid = cleaned_df['user_id'].iloc[0]
sample_iid = cleaned_df['item_id'].iloc[0]

print('\n【Feature Store 在线读取验证】')
print(f'\n  用户 {sample_uid} 特征（取前5个）:')
u = fs.get_user(sample_uid)
for k, v in list(u.items())[:5]: print(f'    {k}: {v}')

print(f'\n  视频 {sample_iid} 特征（取前5个）:')
i = fs.get_item(sample_iid)
for k, v in list(i.items())[:5]: print(f'    {k}: {v}')

print(f'\n  用户 {sample_uid} 序列:')
s = fs.get_seq(sample_uid)
print(f'    seq_len: {s.get("seq_len")}')
print(f'    all_seq: {s.get("all_seq", [])[:10]} ...')

---
## Cell 14 — Pipeline 健康度报告（PSI 监控）

In [ ]:
# ============================================================
# Pipeline 健康度报告
# 对应: 企业在线监控看板（Grafana / 阿里云监控）
#
# PSI（Population Stability Index）— 特征稳定性指数
#   PSI < 0.1  → ✅ 稳定（正常）
#   PSI < 0.2  → ⚠️  轻微漂移（注意）
#   PSI >= 0.2 → ❌ 显著漂移（告警，需调查）
# ============================================================
from scipy.stats import ks_2samp

def compute_psi(a, b, bins=10):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    edges = np.linspace(min(a.min(),b.min()), max(a.max(),b.max()), bins+1)
    a_hist = np.histogram(a, bins=edges)[0]
    b_hist = np.histogram(b, bins=edges)[0]
    a_pct  = (a_hist + 1e-8) / len(a)
    b_pct  = (b_hist + 1e-8) / len(b)
    return float(np.sum((b_pct - a_pct) * np.log(b_pct / a_pct)))

# ---- 汇总报告 ----
print('\n' + '='*65)
print('  📋  STEP 1 PIPELINE 完成报告')
print('='*65)

summary = [
    ['原始评分数据',       f'{len(df_ratings):,}',      '行'],
    ['清洗后数据',         f'{len(cleaned_df):,}',      '行'],
    ['有效用户数',         f'{cleaned_df["user_id"].nunique():,}', '个'],
    ['有效视频数',         f'{cleaned_df["item_id"].nunique():,}', '个'],
    ['训练样本（含负采样）', f'{len(train_samples):,}',  '行'],
    ['测试样本',           f'{len(test_samples):,}',    '行'],
    ['样本特征维度',        f'{train_samples.shape[1]}','列'],
    ['Feature Store Keys', f'{len(fs.store):,}',        '个'],
    ['用户特征数',         f'{user_features.shape[1]}', '列'],
    ['视频特征数',         f'{item_features.shape[1]}', '列'],
]
print(tabulate(summary, headers=['指标','数值','单位'], tablefmt='simple'))

# PSI 监控
print('\n【特征稳定性监控 PSI — 训练集 vs 测试集】')
monitor_feats = ['avg_rating', 'interaction_count', 'positive_rate', 'ctr_proxy']
psi_rows = []
for feat in monitor_feats:
    if feat in train_samples.columns and feat in test_samples.columns:
        psi = compute_psi(train_samples[feat].dropna(), test_samples[feat].dropna())
        status = '✅ 稳定' if psi < 0.1 else ('⚠️  注意' if psi < 0.2 else '❌ 告警')
        psi_rows.append([feat, f'{psi:.4f}', status])
print(tabulate(psi_rows, headers=['特征','PSI','状态'], tablefmt='simple'))

# 输出文件清单
print('\n【输出文件清单】')
file_rows = []
for root, dirs, files in os.walk(BASE_DIR):
    for f in sorted(files):
        fp   = os.path.join(root, f)
        rel  = fp.replace(BASE_DIR + '/', '')
        size = os.path.getsize(fp) / 1024**2
        file_rows.append([rel, f'{size:.2f} MB'])
print(tabulate(file_rows, headers=['文件路径','大小'], tablefmt='simple'))

print('\n' + '='*65)
print('  ✅  Step 1 完成！')
print('  ▶   下一步: Step 2 — 双塔召回模型 (Two-Tower DSSM)')
print('='*65)